### Car Value Prediction Model
##### Yiscah Mark



#### I am uploading the dataset

In [2]:
from google.colab import files

uploaded = files.upload()

Saving CAR DETAILS FROM CAR DEKHO.csv to CAR DETAILS FROM CAR DEKHO.csv


In [4]:
import pandas as pd
df = pd.read_csv("CAR DETAILS FROM CAR DEKHO.csv")

#### Printing the first five lines of data to see it and make sure it uploaded.

In [5]:
df.head()

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner
0,Maruti 800 AC,2007,60000,70000,Petrol,Individual,Manual,First Owner
1,Maruti Wagon R LXI Minor,2007,135000,50000,Petrol,Individual,Manual,First Owner
2,Hyundai Verna 1.6 SX,2012,600000,100000,Diesel,Individual,Manual,First Owner
3,Datsun RediGO T Option,2017,250000,46000,Petrol,Individual,Manual,First Owner
4,Honda Amaze VX i-DTEC,2014,450000,141000,Diesel,Individual,Manual,Second Owner


#### Importing the necesary libraries

In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

#### In the following cell I am setting my variables, y is the selling price, its the target value what I am looking to predict. X, the variables are all the other columns besides for the selling price. I am also excluding, the name column because there are to many unique names (in last model I extracted the company for every model so I had less unique values. I decided to skip that step for now because there are so many other variables, and it won't make such a big difference.)

In [30]:
X = df.drop(columns=['selling_price', 'name'])
y = df['selling_price']

#### now i split the data into 80% to train the model and 20% to see how accurate the model was in guessing the price of a vehicle.

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


### training\ the linear regression model
#### i tried to train it the first time and it didn;t work becuase some of the columsn such as fuel type had a string, so this time i am one-hot encoding them so it can go through.

In [13]:
model = LinearRegression()

# Perform one-hot encoding on categorical columns for X_train and X_test
X_train_encoded = pd.get_dummies(X_train, drop_first=True)
X_test_encoded = pd.get_dummies(X_test, drop_first=True)

missing_cols_in_test = set(X_train_encoded.columns) - set(X_test_encoded.columns)
for c in missing_cols_in_test:
    X_test_encoded[c] = 0

X_test_encoded = X_test_encoded[X_train_encoded.columns]

model.fit(X_train_encoded, y_train)

LinearRegression()

In [14]:
# Make predictions on the encoded test set
y_pred = model.predict(X_test_encoded)

# Evaluate the model using R-squared
r2 = r2_score(y_test, y_pred)
print(f"R-squared Score: {r2:.4f}")

R-squared Score: 0.4031


#### I tested the data and got a very low r-squared value. it means that the model is only accurate 40% of the time.

#### i went back to last week's assignment to see if there was a variable that didn't contribute so much and was just maknig noise, the fuel column didn;t seem important so i redid the model without it, but the r2 value was 32%, even less. so i erased the code.

##### since the model performed poorly, I decided to extract the first word (company) from the name and add it to the model. Also to make an age of car feature wich is a little more understandable than a date.

#### Extracting Company Name from 'name' column, and creating and "age of car" column.

In [38]:
import datetime

current_year = datetime.datetime.now().year
df['age_of_car'] = current_year - df['year']
display(df.head())

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,company,age_of_car
0,Maruti 800 AC,2007,60000,70000,Petrol,Individual,Manual,First Owner,Maruti,19
1,Maruti Wagon R LXI Minor,2007,135000,50000,Petrol,Individual,Manual,First Owner,Maruti,19
2,Hyundai Verna 1.6 SX,2012,600000,100000,Diesel,Individual,Manual,First Owner,Hyundai,14
3,Datsun RediGO T Option,2017,250000,46000,Petrol,Individual,Manual,First Owner,Datsun,9
4,Honda Amaze VX i-DTEC,2014,450000,141000,Diesel,Individual,Manual,Second Owner,Honda,12


In [46]:
# Create a new 'company' column by extracting the first word from the 'name' column
df['company'] = df['name'].apply(lambda x: x.split(' ')[0])
display(df.head())

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,company,age_of_car
0,Maruti 800 AC,2007,60000,70000,Petrol,Individual,Manual,First Owner,Maruti,19
1,Maruti Wagon R LXI Minor,2007,135000,50000,Petrol,Individual,Manual,First Owner,Maruti,19
2,Hyundai Verna 1.6 SX,2012,600000,100000,Diesel,Individual,Manual,First Owner,Hyundai,14
3,Datsun RediGO T Option,2017,250000,46000,Petrol,Individual,Manual,First Owner,Datsun,9
4,Honda Amaze VX i-DTEC,2014,450000,141000,Diesel,Individual,Manual,Second Owner,Honda,12


#### Redefining X (features) to include 'company' and exclude 'name'.

In [47]:
# Redefine X to include 'company' and 'age_of_car', and exclude 'name', 'selling_price', and 'year'
X_final = df.drop(columns=['selling_price', 'name', 'year'])
y_final = df['selling_price']

display(X_final.head())

,km_driven,fuel,seller_type,transmission,owner,company,age_of_car
0,70000,Petrol,Individual,Manual,First Owner,Maruti,19
1,50000,Petrol,Individual,Manual,First Owner,Maruti,19
2,100000,Diesel,Individual,Manual,First Owner,Hyundai,14
3,46000,Petrol,Individual,Manual,First Owner,Datsun,9
4,141000,Diesel,Individual,Manual,Second Owner,Honda,12


#### Re-splitting the data with the new feature set

In [48]:
X_train_final, X_test_final, y_train_final, y_test_final = train_test_split(X_final, y_final, test_size=0.2, random_state=42)

#### Training the linear regression model with the new 'company' feature

In [49]:
model_with_company = LinearRegression()

# Perform one-hot encoding on categorical columns for X_train_final and X_test_final
X_train_encoded_final = pd.get_dummies(X_train_final, drop_first=True)
X_test_encoded_final = pd.get_dummies(X_test_final, drop_first=True)

# Align columns
missing_cols_in_test_final = set(X_train_encoded_final.columns) - set(X_test_encoded_final.columns)
for c in missing_cols_in_test_final:
    X_test_encoded_final[c] = 0
X_test_encoded_final = X_test_encoded_final[X_train_encoded_final.columns]

model_with_company.fit(X_train_encoded_final, y_train_final)

LinearRegression()

#### Evaluating the new model's performance

In [50]:
# Make predictions on the encoded test set
y_pred_with_company = model_with_company.predict(X_test_encoded_final)

# Evaluate the model using R-squared
r2_with_company = r2_score(y_test_final, y_pred_with_company)
print(f"R-squared Score (with 'company' and 'age_of_car' features): {r2_with_company:.4f}")

R-squared Score (with 'company' and 'age_of_car' features): 0.4576


### we are making improvement, the r2 score went up 5%. still not great but better.

#### Adding Polynomial Features for 'km_driven' and 'age_of_car'. this feature can catpture nonlinear relationships.

In [55]:
from sklearn.preprocessing import PolynomialFeatures

numerical_features = ['km_driven', 'age_of_car']

poly = PolynomialFeatures(degree=2, include_bias=False)

# Apply polynomial transformation to the selected numerical features in X_final
X_poly_features = poly.fit_transform(X_final[numerical_features])

# Create a DataFrame with the new polynomial features
X_poly_df = pd.DataFrame(X_poly_features, columns=poly.get_feature_names_out(numerical_features), index=X_final.index)

# Drop original numerical features and concatenate the new polynomial features
X_final_poly = X_final.drop(columns=numerical_features)
X_final_poly = pd.concat([X_final_poly, X_poly_df], axis=1)

display(X_final_poly.head())

,fuel,seller_type,transmission,owner,company,km_driven,age_of_car,km_driven^2,km_driven age_of_car,age_of_car^2
0,Petrol,Individual,Manual,First Owner,Maruti,70000.0,19.0,4.900000e+09,1330000.0,361.0
1,Petrol,Individual,Manual,First Owner,Maruti,50000.0,19.0,2.500000e+09,950000.0,361.0
2,Diesel,Individual,Manual,First Owner,Hyundai,100000.0,14.0,1.000000e+10,1400000.0,196.0
3,Petrol,Individual,Manual,First Owner,Datsun,46000.0,9.0,2.116000e+09,414000.0,81.0
4,Diesel,Individual,Manual,Second Owner,Honda,141000.0,12.0,1.988100e+10,1692000.0,144.0


#### Re-splitting the data with polynomial features

In [56]:
X_train_poly, X_test_poly, y_train_poly, y_test_poly = train_test_split(X_final_poly, y_final, test_size=0.2, random_state=42)

#### Training the linear regression model with polynomial features

In [57]:
model_poly = LinearRegression()

# Perform one-hot encoding on categorical columns for X_train_poly and X_test_poly
X_train_encoded_poly = pd.get_dummies(X_train_poly, drop_first=True)
X_test_encoded_poly = pd.get_dummies(X_test_poly, drop_first=True)

# Align columns
missing_cols_in_test_poly = set(X_train_encoded_poly.columns) - set(X_test_encoded_poly.columns)
for c in missing_cols_in_test_poly:
    X_test_encoded_poly[c] = 0
X_test_encoded_poly = X_test_encoded_poly[X_train_encoded_poly.columns]

model_poly.fit(X_train_encoded_poly, y_train_poly)

LinearRegression()

#### Evaluating the new model's performance with polynomial features

In [58]:
# Make predictions on the encoded test set
y_pred_poly = model_poly.predict(X_test_encoded_poly)

# Evaluate the model using R-squared
r2_poly = r2_score(y_test_poly, y_pred_poly)
print(f"R-squared Score (with 'company', 'age_of_car', and polynomial features): {r2_poly:.4f}")

R-squared Score (with 'company', 'age_of_car', and polynomial features): 0.5094


### My model isn't super accurate but I learned alo tas i tweaked the model. Overall i am happy that i was able to move the r2 score up over 10%.